Download all the NICE guidelines linked from https://www.rcpch.ac.uk/resources/clinical-guideline-directory so we can upload them into pAIr

In [10]:
import httpx
from bs4 import BeautifulSoup

client = httpx.Client()
page = client.get("https://www.rcpch.ac.uk/resources/clinical-guideline-directory")

soup = BeautifulSoup(page.text, "html.parser")

In [15]:
links = soup.select(".body-box li > a")
links = [(a.text, a["href"]) for a in links if a["href"].startswith("http")]
links

[('Rehabilitation for chronic neurological disorders including acquired brain injury ',
  'https://www.nice.org.uk/guidance/ng252'),
 ('Pneumonia: diagnosis and management',
  'https://www.nice.org.uk/guidance/ng250'),
 ('Pneumonia: diagnosis and management',
  'https://www.nice.org.uk/guidance/qs110'),
 ('Atopic eczema in under 12s: diagnosis and management',
  'https://www.nice.org.uk/guidance/cg57'),
 ('Suspected acute respiratory infection in over 16s: assessment at first presentation and initial management ',
  'https://www.nice.org.uk/guidance/ng237/chapter/Update-information'),
 ('Overweight and obesity management - quality standard',
  'https://www.nice.org.uk/guidance/qs212'),
 ('Headaches in over 12s: diagnosis and management ',
  'https://www.nice.org.uk/guidance/cg150'),
 ('Diagnosis and management of pituitary adenomas in childhood and adolescence',
  'https://www.cclg.org.uk/guidelines/pituitary-adenomas'),
 ('Maternal and child nutrition: nutrition and weight management 

In [18]:
from urllib.parse import urlparse

import pandas as pd

df = pd.DataFrame(links, columns=["title", "url"])
df["domain"] = df["url"].apply(lambda u: urlparse(u).hostname)
df

df["domain"].value_counts()

domain
www.nice.org.uk         434
www.sign.ac.uk           17
www.cclg.org.uk          11
www.rcpch.ac.uk           9
www.rcophth.ac.uk         4
www.bbuk.org.uk           3
brit-thoracic.org.uk      2
www.rcog.org.uk           2
www.rcot.co.uk            2
www.chelwest.nhs.uk       2
www.headsmart.org.uk      2
www.dsmig.org.uk          1
www.acb.org.uk            1
www.resus.org.uk          1
www.bsaci.org             1
www.rcpsych.ac.uk         1
ukkidney.org              1
www.rcr.ac.uk             1
Name: count, dtype: int64

In [20]:
nice_links = df[df["domain"] == "www.nice.org.uk"]["url"].tolist()
nice_links

['https://www.nice.org.uk/guidance/ng252',
 'https://www.nice.org.uk/guidance/ng250',
 'https://www.nice.org.uk/guidance/qs110',
 'https://www.nice.org.uk/guidance/cg57',
 'https://www.nice.org.uk/guidance/ng237/chapter/Update-information',
 'https://www.nice.org.uk/guidance/qs212',
 'https://www.nice.org.uk/guidance/cg150',
 'https://www.nice.org.uk/guidance/ng247',
 'https://www.nice.org.uk/guidance/ng246',
 'https://www.nice.org.uk/guidance/ng245/chapter/Rationale-and-impact',
 'https://www.nice.org.uk/guidance/qs75',
 'https://www.nice.org.uk/guidance/qs39',
 'https://www.nice.org.uk/guidance/ng87',
 'https://www.nice.org.uk/guidance/cg100',
 'https://www.nice.org.uk/guidance/cg115',
 'https://www.nice.org.uk/guidance/qs204',
 'https://www.nice.org.uk/guidance/cg134',
 'https://www.nice.org.uk/guidance/ng243',
 'https://www.nice.org.uk/guidance/qs121',
 'https://www.nice.org.uk/guidance/ng15',
 'https://www.nice.org.uk/guidance/cg57',
 'https://www.nice.org.uk/guidance/qs97',
 'htt

In [31]:
import os

for link in nice_links:
    nice_page = client.get(link)
    nice_soup = BeautifulSoup(nice_page.text, "html.parser")

    pdf_link = nice_soup.select_one("[data-track='guidancedownload']")

    if not pdf_link:
        print("\tSkipped - no download button")
    else:
        pdf_link = "https://www.nice.org.uk" + pdf_link['href']
        local_filename = "../source_docs/" + pdf_link.split('/')[-1] + ".pdf"

        if os.path.exists(local_filename):
            print(f"\tSkipping as already downloaded")
        else:
            print(f"\t{local_filename}")
            with client.stream("GET", pdf_link) as r:
                with open(local_filename, 'wb') as f:
                    for chunk in r.iter_bytes():
                        f.write(chunk)

	../source_docs/rehabilitation-for-chronic-neurological-disorders-including-acquired-brain-injury-pdf-66144013706437.pdf
	../source_docs/pneumonia-diagnosis-and-management-pdf-66144010347205.pdf
	../source_docs/pneumonia-diagnosis-and-management-pdf-75545291391685.pdf
	../source_docs/atopic-eczema-in-under-12s-diagnosis-and-management-pdf-975512529349.pdf
	../source_docs/suspected-acute-respiratory-infection-in-over-16s-assessment-at-first-presentation-and-initial-management-pdf-66143901172165.pdf
	../source_docs/overweight-and-obesity-management-pdf-75547471533253.pdf
	../source_docs/headaches-in-over-12s-diagnosis-and-management-pdf-35109624582853.pdf
	../source_docs/maternal-and-child-nutrition-nutrition-and-weight-management-in-pregnancy-and-nutrition-in-children-up-to-5-years-pdf-66143961638341.pdf
	../source_docs/overweight-and-obesity-management-pdf-66143959958725.pdf
	../source_docs/asthma-diagnosis-monitoring-and-chronic-asthma-management-bts-nice-sign-pdf-66143958279109.pdf
	